# Pipeline

> Run OCR + fix the markdown headings + describe images/figures

In [ ]:
#| default_exp pipeline

In [ ]:
#| export
from fastcore.all import *
from mistocr.core import read_pgs
#from re import sub, findall, MULTILINE
#from pydantic import BaseModel
#from lisette import *
#from lisette.core import completion
#from typing import Callable
import os
import json
import shutil
from asyncio import Semaphore, gather, sleep

In [ ]:
#| export
@delegates(add_img_descs)
async def pdf_to_md(pdf_path:str, dst:str, ocr_output:str=None, model:str='claude-sonnet-4-5', add_img_desc:bool=True, progress:bool=True, **kwargs):
    "Convert PDF to markdown with fixed headings and image descriptions"
    from mistocr.core import ocr_pdf
    ocr_dir = Path(ocr_output) if ocr_output else Path(pdf_path).with_suffix('')
    n_steps = 3 if add_img_desc else 2
    if progress: print(f"Step 1/{n_steps}: Running OCR on {pdf_path}...")
    ocr_pdf(pdf_path, ocr_dir)
    if progress: print(f"Step 2/{n_steps}: Fixing heading hierarchy...")
    fix_hdgs(ocr_dir, model=model)
    if add_img_desc:
        if progress: print(f"Step 3/{n_steps}: Adding image descriptions...")
        await add_img_descs(ocr_dir, dst=dst, model=model, progress=progress, **kwargs)
    elif dst and Path(dst) != ocr_dir:
        shutil.copytree(ocr_dir, dst, dirs_exist_ok=True)
    if progress: print("Done!")